In [ ]:
import pandas as pd
import numpy as np

# %% [code]
# Load Venmo statement CSV
venmo_df = pd.read_csv('venmo_statement.csv')

# Load All Bets CSV
bets_df = pd.read_csv('all_bets.csv')

# %% [markdown]
# ## 2. Inspect & Clean the Venmo Data
# 
# For this example, we expect the Venmo statement to have at least these columns:
# 
# - **ID**, **Datetime**, **Type**, **Status**, **Note**, **From**, **To**, **Amount (total)**
# 
# We will rename some columns for convenience.

# %% [code]
# Show the columns in the Venmo CSV
print("Venmo Statement Columns:")
print(venmo_df.columns)

venmo_df.rename(columns={
    'Datetime': 'Timestamp',
    'Amount (total)': 'Amount',
    'From': 'FromUser',
    'To': 'ToUser'
}, inplace=True)

venmo_df['Timestamp'] = pd.to_datetime(venmo_df['Timestamp'], errors='coerce')

venmo_df.head()

donation_mask = (venmo_df['Amount'] == 5) & (
    venmo_df['Note'].str.contains('donation|entry|fee', case=False, na=False)
)
donations_df = venmo_df[donation_mask].copy()

print("Found Donation Transactions:")
print(donations_df[['Timestamp', 'FromUser', 'ToUser', 'Amount', 'Note']].head())

bet_mask = venmo_df['Note'].str.contains('bet', case=False, na=False)
bet_trans_df = venmo_df[bet_mask].copy()

print("\nFound Bet Transactions:")
print(bet_trans_df[['Timestamp', 'FromUser', 'ToUser', 'Amount', 'Note']].head())

unique_handles = bets_df['Venmo Handle'].unique()
print("Unique handles in bets CSV:", unique_handles)

missing_donations = []

for handle in unique_handles:
    match_found = donations_df['FromUser'].str.contains(handle, case=False, na=False).any()
    if not match_found:
        missing_donations.append(handle)

print("\nThe following handles have NO matching $5 donation transaction:")
print(missing_donations)
bets_df.rename(columns={'Venmo Handle': 'Handle'}, inplace=True)

merged_bets = pd.merge(
    bets_df,
    bet_trans_df,
    left_on=['Handle', 'Amount'],  # We match on handle and amount. Adjust if needed.
    right_on=['FromUser', 'Amount'],
    how='left', 
    indicator=True
)

missing_bet_trans = merged_bets[merged_bets['_merge'] == 'left_only']

print("\nBets with NO matching Venmo transaction (possible missing bet payment):")
print(missing_bet_trans[['Bet ID', 'Handle', 'Match Name', 'Competitor Chosen', 'Amount']])

missing_donations_df = pd.DataFrame(missing_donations, columns=['Missing Donation Handle'])
missing_donations_df.to_csv('missing_donations.csv', index=False)

missing_bet_trans.to_csv('missing_bet_transactions.csv', index=False)

print("\nMissing donations and missing bet transactions have been saved to CSV files.")
